# Lark: BFS or DFS?

## Parsing Phase
Lark **does not use BFS or DFS** during parsing.

It uses parsing algorithms:
- **LALR** — shift/reduce, table-driven  
- **Earley** — chart parsing (dynamic programming)  
- **CYK** — matrix-based dynamic programming  

None of these are BFS or DFS.

---

## Tree Traversal Phase
When Lark walks the parse tree (e.g., with `Transformer` or `Visitor`):

- **Traversal = DFS (Depth-First Search)**

Because nodes are visited recursively.

---

## Summary Table

| Component            | BFS/DFS?          |
|----------------------|--------------------|
| LALR Parser          | ❌ Neither         |
| Earley Parser        | ❌ Neither         |
| CYK Parser           | ❌ Neither         |
| Parse Tree Traversal | ✔ DFS             |


## What is Lark?

**Lark** is a modern parsing library for Python, capable of parsing any **context-free grammar**.  
It is designed to handle a wide range of parsing tasks while maintaining **high performance** and **flexibility**.

Whether you need to parse:
- Simple mathematical expressions
- Domain-specific languages (DSLs)
- Complex programming languages  

Lark makes the job **easier, cleaner, and more maintainable**.


# Trading Strategy DSL Program Explanation

**Location:** `D:\Projects_Main\Finacal_Adviser\nlp-to-strategy-engine\dsl`

## Overview

This DSL converts human trading rules into executable code. Think of it as a **translator** between English-like strategy descriptions and computer-understandable instructions.

---

## The 5 Core Files

### 1. **grammar.lark** - The Language Rules

Defines what's "legal" to write in this language, like English grammar rules.

**Structure:**
```
strategy → ENTRY + optional EXIT
rule_block → comparisons connected by AND/OR
comparison → left operator right
```

**Supports:**
- **Operators:** `>`, `<`, `>=`, `<=`, `==`, `!=`, `crosses_above`, `crosses_below`
- **Data:** `close`, `open`, `high`, `low`, `volume`
- **Indicators:** `sma()`, `ema()`, `rsi()`, `macd()`, `bb_upper()`, etc.
- **Numbers:** `100`, `1.5K` (thousand), `2M` (million), `1B` (billion)
- **Time refs:** `close_prev`, `high_5d_ago`

---

### 2. **ast_nodes.py** - Data Structures

Defines building blocks (like Lego pieces) that represent different parts of a strategy.

**Node Types:**
- **Strategy** - Complete strategy with entry/exit
- **Comparison** - `close > 100` 
- **BooleanOp** - AND/OR connections
- **Indicator** - `sma(close, 20)`
- **Series** - `close`, `volume`
- **Number** - `50000`
- **TimeReference** - `close_prev`

**Example AST Structure:**
```
Strategy
├── entry: Comparison
│   ├── left: Indicator(rsi, 14)
│   ├── operator: ">"
│   └── right: Number(70)
└── exit: Comparison
    ├── left: Indicator(rsi, 14)
    ├── operator: "<"
    └── right: Number(30)
```

---

### 3. **parser.py** - The Translator

Reads DSL text and builds the AST (Abstract Syntax Tree).

**Process:**
1. **Input:** `"ENTRY: rsi(close, 14) > 70"`
2. **Lark parses** using grammar rules
3. **DSLTransformer** converts to AST nodes
4. **Output:** Strategy object with structured data

**Key Components:**
- **DSLParser** - Main parser class, loads grammar, parses text
- **DSLTransformer** - Converts parse tree → AST nodes
- **Handles:** Indicators, comparisons, boolean logic, numbers with scales (K/M/B)

---

### 4. **validator.py** - Quality Control

Checks if the strategy makes sense.

**Validates:**
- Series names are valid (`close`, `open`, etc.)
- Operators are correct (`>`, `crosses_above`, etc.)
- Indicators have right parameters (`sma` needs 2 params)
- Entry block exists (mandatory)

**Quality Metrics:**
- **Complexity score** - How many conditions (simple=1, complex=5+)
- **Indicator count** - How many technical indicators used
- **Warnings** - "No exit defined", "Too complex"

---

### 5. **__init__.py** - Package Exports

Makes all classes/functions accessible with simple imports.

```python
from dsl import parse_dsl, validate_dsl, get_strategy_quality
```

---

## How It Works (End-to-End)

### **Step 1: Input (from NLP)**
```python
ParsedStrategy(
    rule=TradingRule(
        entry=[Condition(left="rsi(close,14)", operator=">", right=70)],
        exit=[Condition(left="rsi(close,14)", operator="<", right=30)]
    )
)
```

### **Step 2: Convert to DSL Text**
```
ENTRY: rsi(close, 14) > 70
EXIT: rsi(close, 14) < 30
```

### **Step 3: Parse to AST**
```python
ast = parse_dsl(dsl_text)
# Creates Strategy object with structured nodes
```

### **Step 4: Validate**
```python
is_valid, errors, warnings = validate_dsl(ast)
# Checks correctness, returns issues
```

### **Step 5: Quality Check**
```python
quality = get_strategy_quality(ast)
# Returns: complexity=1, indicators=1, has_exit=True
```

---

## Why This Design?

| Feature | Benefit |
|---------|---------|
| **Separation of Concerns** | Grammar, parsing, validation are separate |
| **Extensible** | Easy to add new indicators/operators |
| **Type-Safe** | Pydantic-like validation with dataclasses |
| **Debuggable** | Clear AST structure for inspection |
| **Maintainable** | Grammar changes don't affect validator |

---

## Real-World Flow

```mermaid
graph LR
    A[Human Input] --> B[NLP Parser]
    B --> C[ParsedStrategy]
    C --> D[DSL Converter]
    D --> E[DSL Text]
    E --> F[DSL Parser]
    F --> G[AST]
    G --> H[Validator]
    H --> I[Code Generator]
    I --> J[Executable Strategy]
```

**The DSL sits between NLP understanding and code execution**, ensuring strategies are correctly structured before generating trading code.

---

## Example Usage

### Input Strategy
```
"Buy when RSI crosses above 70 with $50k at 50% position. Sell when RSI drops below 30."
```

### Generated DSL
```
ENTRY: rsi(close, 14) > 70
EXIT: rsi(close, 14) < 30
```

### Parsed AST
```python
Strategy(
    entry=Comparison(
        left=Indicator(name='rsi', params=['close', 14]),
        operator='>',
        right=Number(value=70)
    ),
    exit=Comparison(
        left=Indicator(name='rsi', params=['close', 14]),
        operator='<',
        right=Number(value=30)
    )
)
```

### Validation Result
```python
{
    'valid': True,
    'errors': [],
    'warnings': [],
    'complexity': 1,
    'indicators': ['rsi'],
    'has_exit': True
}
```

---

## File Relationships

```
dsl/
├── grammar.lark          ← Defines syntax rules
├── ast_nodes.py          ← Data structures
├── parser.py             ← Text → AST converter (uses grammar.lark)
├── validator.py          ← AST validator (uses ast_nodes.py)
└── __init__.py           ← Exports everything
```

**Flow:** `grammar.lark` → `parser.py` → `ast_nodes.py` → `validator.py`

---

## Quick Start

```python
from dsl import parse_dsl, validate_dsl, get_strategy_quality

# Parse DSL text
dsl_text = "ENTRY: close > sma(close, 20)\nEXIT: rsi(close, 14) > 70"
ast = parse_dsl(dsl_text)

# Validate
is_valid, errors, warnings = validate_dsl(ast)

# Check quality
quality = get_strategy_quality(ast)
print(f"Complexity: {quality['entry_complexity']}")
print(f"Indicators: {quality['indicator_count']}")
```

---

## Summary

The DSL module provides a **robust, type-safe way** to represent trading strategies as structured data. It bridges the gap between natural language input and executable code, ensuring strategies are syntactically correct and semantically valid before execution.
